# Fund Performance Analytics
**Day 4 | Capstone Project I – Mutual Fund Analytics**

Computes from-scratch:
- Daily Returns & distribution validation
- CAGR (1yr / 3yr / 5yr) comparison table
- Sharpe Ratio (Rf = 6.5%) — ranked
- Sortino Ratio (downside deviation only)
- Alpha & Beta via OLS regression on NIFTY100
- Maximum Drawdown + worst drawdown date range
- Fund Scorecard (0–100) composite ranking
- Benchmark comparison chart + tracking error

**Deliverables:** `fund_scorecard.csv`, `alpha_beta.csv`, 3 chart PNGs

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.1)

PROC   = '../data/processed'
RAW    = '../data/raw'
CHARTS = '../reports/charts'
os.makedirs(CHARTS, exist_ok=True)

RF_ANNUAL    = 0.065
RF_DAILY     = RF_ANNUAL / 252
TRADING_DAYS = 252

BLUE='#2563EB'; ORANGE='#F97316'; GREEN='#16A34A'
RED='#DC2626';  PURPLE='#7C3AED'; GREY='#6B7280'; TEAL='#0D9488'
print('Setup complete')

In [ ]:
fm    = pd.read_csv(f'{PROC}/01_fund_master.csv')
nav   = pd.read_csv(f'{RAW}/02_nav_history.csv', parse_dates=['date'])
bench = pd.read_csv(f'{PROC}/10_benchmark_indices.csv', parse_dates=['date'])

nav = nav.sort_values(['amfi_code','date']).reset_index(drop=True)
nifty100  = bench[bench['index_name']=='NIFTY100'].set_index('date')['close_value'].sort_index()
nifty50   = bench[bench['index_name']=='NIFTY50'].set_index('date')['close_value'].sort_index()
bench_ret100 = nifty100.pct_change().dropna()
bench_ret50  = nifty50.pct_change().dropna()
last_date    = nav['date'].max()
print(f'NAV: {nav.shape} | Benchmark indices: {bench["index_name"].nunique()}')

---
## 1. Daily Returns
`daily_return = nav_t / nav_t-1 - 1` computed for all 40 schemes.

In [ ]:
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
returns_wide = (nav.dropna(subset=['daily_return'])
                   .pivot(index='date', columns='amfi_code', values='daily_return'))

r = nav['daily_return'].dropna()
print(f'Mean daily return : {r.mean()*100:.4f}%')
print(f'Std daily return  : {r.std()*100:.4f}%')
print(f'Min / Max         : {r.min()*100:.2f}% / {r.max()*100:.2f}%')
print(f'Negative days     : {(r<0).mean()*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(r*100, bins=120, color=BLUE, alpha=0.7, density=True, edgecolor='none')
ax.axvline(0, color=RED, linewidth=1.5, linestyle='--')
ax.set_xlabel('Daily Return (%)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Distribution of Daily Returns — All 40 Funds', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 2. CAGR — 1yr, 3yr, 5yr
`CAGR = (NAV_end / NAV_start)^(1/n) - 1`

In [ ]:
def get_nav_on(df_fund, target_date):
    sub = df_fund[df_fund['date'] <= target_date]
    return sub.iloc[-1]['nav'] if not sub.empty else np.nan

def cagr(nav_end, nav_start, years):
    if np.isnan(nav_start) or nav_start <= 0: return np.nan
    return (nav_end / nav_start) ** (1/years) - 1

rows = []
for code, grp in nav.groupby('amfi_code'):
    grp = grp.sort_values('date')
    end = grp.iloc[-1]['nav']; edt = grp.iloc[-1]['date']
    rows.append({'amfi_code': code,
                 'cagr_1yr': cagr(end, get_nav_on(grp, edt-pd.DateOffset(years=1)), 1),
                 'cagr_3yr': cagr(end, get_nav_on(grp, edt-pd.DateOffset(years=3)), 3),
                 'cagr_5yr': cagr(end, get_nav_on(grp, edt-pd.DateOffset(years=5)), 5)})

cagr_df = pd.DataFrame(rows)
merged  = cagr_df.merge(fm[['amfi_code','scheme_name','plan','sub_category']], on='amfi_code')
print(merged[['scheme_name','plan','cagr_1yr','cagr_3yr','cagr_5yr']]
      .sort_values('cagr_3yr', ascending=False)
      .head(10).to_string(index=False))

---
## 3. Sharpe Ratio
`Sharpe = (Mean(Rp) - Rf) / Std(Rp) * sqrt(252)` | Rf = 6.5%

In [ ]:
def sharpe(r):
    r = r.dropna()
    if len(r) < 30: return np.nan
    return ((r - RF_DAILY).mean() / r.std()) * np.sqrt(TRADING_DAYS)

sharpe_s = returns_wide.apply(sharpe).rename('sharpe_ratio')
sharpe_df = sharpe_s.reset_index().merge(fm[['amfi_code','scheme_name','plan']], on='amfi_code')
print('Top 10 by Sharpe:')
print(sharpe_df.sort_values('sharpe_ratio', ascending=False)[['scheme_name','plan','sharpe_ratio']].head(10).to_string(index=False))

---
## 4. Sortino Ratio
Same as Sharpe but denominator = downside std (negative return days only).

In [ ]:
def sortino(r):
    r = r.dropna()
    if len(r) < 30: return np.nan
    downside = r[r < 0]
    if len(downside) < 2: return np.nan
    return ((r.mean() - RF_DAILY) / downside.std()) * np.sqrt(TRADING_DAYS)

sortino_s = returns_wide.apply(sortino).rename('sortino_ratio')
print(f'Sortino range: {sortino_s.min():.3f} to {sortino_s.max():.3f}')
print(f'Sortino median: {sortino_s.median():.3f}')

---
## 5. Alpha & Beta (OLS regression vs NIFTY100)
`Beta = slope | Alpha = intercept x 252` (annualised)

In [ ]:
ab_rows = []
for code in returns_wide.columns:
    fund_r = returns_wide[code].dropna()
    common = fund_r.index.intersection(bench_ret100.index)
    if len(common) < 60:
        ab_rows.append({'amfi_code':code,'beta':np.nan,'alpha_annual':np.nan,'r_squared':np.nan,'tracking_error_nifty100':np.nan})
        continue
    y, x = fund_r.loc[common].values, bench_ret100.loc[common].values
    slope, intercept, r, p, se = stats.linregress(x, y)
    te = np.std(y - x) * np.sqrt(TRADING_DAYS)
    ab_rows.append({'amfi_code':code, 'beta':round(slope,4),
                    'alpha_annual':round(intercept*TRADING_DAYS*100,4),
                    'r_squared':round(r**2,4), 'tracking_error_nifty100':round(te*100,4)})

ab_df = pd.DataFrame(ab_rows)
ab_out = ab_df.merge(fm[['amfi_code','fund_house','scheme_name','plan']], on='amfi_code')
ab_out.to_csv('../alpha_beta.csv', index=False)
print('Top 5 by Alpha:')
print(ab_out.sort_values('alpha_annual',ascending=False)[['scheme_name','plan','beta','alpha_annual','r_squared']].head(5).to_string(index=False))

---
## 6. Maximum Drawdown
`MaxDD = min(NAV / running_max - 1)` per fund, with worst drawdown date range.

In [ ]:
dd_rows = []
for code, grp in nav.groupby('amfi_code'):
    s = grp.sort_values('date').set_index('date')['nav']
    running_max  = s.cummax()
    drawdown     = s / running_max - 1
    max_dd       = drawdown.min()
    trough       = drawdown.idxmin()
    peak         = s.loc[:trough].idxmax()
    dd_rows.append({'amfi_code':code, 'max_drawdown_pct':round(max_dd*100,4),
                    'peak_date':peak.date(), 'trough_date':trough.date(),
                    'dd_duration_days':(trough-peak).days})

dd_df = pd.DataFrame(dd_rows)
worst = dd_df.loc[dd_df['max_drawdown_pct'].idxmin()]
print(f'Worst drawdown: {worst["max_drawdown_pct"]:.2f}%')
print(f'  Peak: {worst["peak_date"]} | Trough: {worst["trough_date"]} | Duration: {worst["dd_duration_days"]} days')
print(f'Median drawdown: {dd_df["max_drawdown_pct"].median():.2f}%')

---
## 7. Fund Scorecard (0–100)
`30% x 3yr_return_rank + 25% x Sharpe_rank + 20% x Alpha_rank + 15% x Expense_rank(inv) + 10% x MaxDD_rank(inv)`

In [ ]:
perf = (fm[['amfi_code','fund_house','scheme_name','category','sub_category','plan','expense_ratio_pct']]
        .merge(cagr_df, on='amfi_code')
        .merge(sharpe_s.reset_index(), on='amfi_code')
        .merge(sortino_s.reset_index(), on='amfi_code')
        .merge(ab_df, on='amfi_code')
        .merge(dd_df[['amfi_code','max_drawdown_pct','peak_date','trough_date','dd_duration_days']], on='amfi_code'))

def pct_rank(s, asc=True):
    return s.rank(ascending=asc, pct=True) * 100

perf['rank_3yr']    = pct_rank(perf['cagr_3yr'])
perf['rank_sharpe'] = pct_rank(perf['sharpe_ratio'])
perf['rank_alpha']  = pct_rank(perf['alpha_annual'])
perf['rank_exp_inv']= pct_rank(perf['expense_ratio_pct'], asc=False)
perf['rank_dd_inv'] = pct_rank(perf['max_drawdown_pct'],  asc=False)
perf['fund_score']  = (0.30*perf['rank_3yr'] + 0.25*perf['rank_sharpe'] +
                       0.20*perf['rank_alpha']+ 0.15*perf['rank_exp_inv']+ 0.10*perf['rank_dd_inv']).round(2)
perf['score_rank']  = perf['fund_score'].rank(ascending=False, method='min').astype(int)
perf = perf.sort_values('fund_score', ascending=False).reset_index(drop=True)

scorecard_cols = ['score_rank','fund_score','amfi_code','fund_house','scheme_name','category',
                  'sub_category','plan','cagr_1yr','cagr_3yr','cagr_5yr','sharpe_ratio',
                  'sortino_ratio','expense_ratio_pct','max_drawdown_pct','peak_date','trough_date',
                  'rank_3yr','rank_sharpe','rank_alpha','rank_exp_inv','rank_dd_inv']
perf[scorecard_cols].to_csv('../fund_scorecard.csv', index=False)
print('Saved fund_scorecard.csv')
print(perf[['scheme_name','plan','fund_score','cagr_3yr','sharpe_ratio','alpha_annual','max_drawdown_pct']].head(10).to_string(index=False))

---
## 8. Benchmark Comparison — Top 5 Equity Funds vs NIFTY50 & NIFTY100

In [ ]:
top5      = perf[perf['category']=='Equity'].head(5)
date_3y   = last_date - pd.DateOffset(years=3)
colors_f  = [BLUE, ORANGE, GREEN, PURPLE, TEAL]

fig, axes = plt.subplots(2, 1, figsize=(14,12), gridspec_kw={'height_ratios':[3,1]})
ax = axes[0]
te_rows = []
for i, (_, row) in enumerate(top5.iterrows()):
    code = row['amfi_code']
    name = row['scheme_name'].split(' - ')[0].strip()
    df   = nav[(nav['amfi_code']==code)&(nav['date']>=date_3y)].sort_values('date')
    if df.empty: continue
    indexed = df['nav'] / df.iloc[0]['nav'] * 100
    ax.plot(df['date'], indexed, label=name, color=colors_f[i], linewidth=2)
    fund_r = df.set_index('date')['nav'].pct_change().dropna()
    common = fund_r.index.intersection(bench_ret100.index)
    if len(common) > 10:
        te = np.std((fund_r.loc[common] - bench_ret100.loc[common]).values) * np.sqrt(TRADING_DAYS) * 100
        te_rows.append({'Fund': name, 'Tracking Error vs NIFTY100 (%)': round(te,2)})

for idx_name, series, color, ls in [('NIFTY 50',nifty50,RED,'--'),('NIFTY 100',nifty100,GREY,':')]:
    s = series[series.index >= date_3y]
    ax.plot(s.index, s/s.iloc[0]*100, label=idx_name, color=color, linewidth=2, linestyle=ls)

ax.set_title('Top 5 Equity Funds vs NIFTY 50 & NIFTY 100 — 3-Year Indexed Performance',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Indexed Value (Base = 100)'); ax.legend(loc='upper left', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f'))
ax.grid(True, alpha=0.3)

ax2 = axes[1]
te_df = pd.DataFrame(te_rows)
bars  = ax2.barh(te_df['Fund'], te_df['Tracking Error vs NIFTY100 (%)'],
                 color=colors_f[:len(te_df)], height=0.5)
for bar, val in zip(bars, te_df['Tracking Error vs NIFTY100 (%)']):
    ax2.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2, f'{val:.2f}%', va='center', fontsize=9)
ax2.set_xlabel('Annualised Tracking Error vs NIFTY100 (%)')
ax2.set_title('Tracking Error (3-Year Window)', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'{CHARTS}/16_benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tracking Errors:')
print(te_df.to_string(index=False))

---
## 9. Fund Scorecard Visual — All 40 Funds

In [ ]:
fig2, ax3 = plt.subplots(figsize=(14,9))
colors_bar  = [GREEN if s>=70 else (ORANGE if s>=50 else RED) for s in perf['fund_score']]
short_names = perf['scheme_name'].str.extract(r'^([^-]+)')[0].str.strip().str[:30]
bars = ax3.barh(short_names[::-1], perf['fund_score'][::-1], color=colors_bar[::-1], height=0.7)
ax3.axvline(50, color=GREY, linestyle='--', linewidth=1)
ax3.set_xlabel('Fund Score (0–100)', fontsize=11)
ax3.set_title('Fund Scorecard — All 40 Schemes\n(30% 3yr CAGR | 25% Sharpe | 20% Alpha | 15% Exp Ratio | 10% Max DD)',
              fontsize=12, fontweight='bold')
ax3.set_xlim(0, 110)
for bar, score in zip(bars[::-1], perf['fund_score']):
    ax3.text(score+0.5, bar.get_y()+bar.get_height()/2, f'{score:.1f}', va='center', fontsize=8)
patches = [mpatches.Patch(color=GREEN, label='>=70 Strong'),
           mpatches.Patch(color=ORANGE,label='50-70 Average'),
           mpatches.Patch(color=RED,   label='<50 Weak')]
ax3.legend(handles=patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(f'{CHARTS}/17_fund_scorecard_bar.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary — Key Performance Findings

| Metric | Best | Range |
|---|---|---|
| 3yr CAGR | Small Cap funds ~23% | 5% – 23% |
| Sharpe Ratio | Liquid/Low-risk funds ~1.4 | -0.8 – 1.4 |
| Alpha (vs NIFTY100) | Small/Mid Cap ~30% pa | 3% – 30% |
| Max Drawdown | Best: -11% (Large Cap) | -11% – -53% |
| Top Fund Score | ICICI Pru Midcap (84.5) | 19.8 – 84.5 |

**Key insight:** Mid-cap funds score highest on the composite scorecard — strong 3yr CAGR + alpha while maintaining moderate drawdowns vs small-cap peers.